In [1]:
import torch
from tqdm import tqdm

from tts.config.stage1.data_config import DataConfig
from tts.config.utils.io import load_config
from tts.data.tts_datafactory import TTSDataFactory

/home/blue2959/monotonic_tts/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_CONFIG_PATH = "/home/blue2959/monotonic_tts/runs/monotonic_tts_libritts+k=1+dec_pos_enc+dur_pred_20260522-102627/data_config.json"

cfg = load_config(DATA_CONFIG_PATH, DataConfig)

factory = TTSDataFactory(cfg)
ds = factory.train_dataset

[Info] Parsed 149736 items from libritts
[Info] Filtering items by duration (1.5s ~ 17.5s)...


Filtering data: 100%|██████████| 149736/149736 [00:21<00:00, 7113.35it/s]


[Info] Filtered 22543 items. Remaining: 127193
[Info] Data split complete: 125922 train, 1271 valid


In [4]:
bad = []
stats = []

for i in tqdm(range(len(ds))):
    try:
        x, y, cond, text = ds[i]

        x_len = int(x.numel())
        y_len = int(y.shape[-1])  # y: (n_mels, T_mel)
        ratio = y_len / max(x_len, 1)
        pair_area = x_len * y_len

        reasons = []

        if x_len <= 0:
            reasons.append("empty_text_tokens")
        if y_len <= 0:
            reasons.append("empty_mel")
        if y_len < x_len:
            reasons.append("mel_len < text_len")
        if ratio < 1.05:
            reasons.append(f"low_frames_per_token={ratio:.3f}")
        if not torch.isfinite(x.float()).all():
            reasons.append("nonfinite_text")
        if not torch.isfinite(y).all():
            reasons.append("nonfinite_mel")
        if not torch.isfinite(cond).all():
            reasons.append("nonfinite_cond")
        if y.ndim != 2:
            reasons.append(f"bad_mel_ndim={y.ndim}")
        if cond.ndim != 1:
            reasons.append(f"bad_cond_ndim={cond.ndim}")

        stats.append((pair_area, x_len, y_len, ratio, i, text[:200]))

        if reasons:
            bad.append({
                "idx": i,
                "x_len": x_len,
                "y_len": y_len,
                "ratio": ratio,
                "pair_area": pair_area,
                "reasons": reasons,
                "text": text,
            })

    except Exception as e:
        bad.append({
            "idx": i,
            "reasons": ["exception"],
            "error": repr(e),
        })

len(bad), bad[:5]

  5%|▍         | 5993/125922 [01:53<37:58, 52.64it/s]


KeyboardInterrupt: 